In [1]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain.agents.middleware import wrap_model_call
import subprocess
from pathlib import Path
import sys

load_dotenv()

C:\Users\ZENOID\AppData\Local\Temp\ipykernel_6872\3746079999.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
c:\Users\ZENOID\Desktop\Home\home\self_made.projects\Standard Projects\code_debugger\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1580.73it/s]


In [3]:
django_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="django_docs",
    embedding_function=embeddings
)

python_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="python_scripts",
    embedding_function=embeddings
)

In [4]:
django_db= django_vectorstore.get()
python_db=python_vectorstore.get()

django_splits=[
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(django_db["documents"], django_db["metadatas"])
]

python_splits = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(python_db["documents"], python_db["metadatas"])
]

django_retriever= django_vectorstore.as_retriever(search_kwargs={"k":4})
python_retriever= python_vectorstore.as_retriever(search_kwargs={"k":4})


all_splits= django_splits + python_splits
print(f"Loaded {len(all_splits)} total splits ({len(django_splits)} Django docs + {len(python_splits)} Python codebase).")
bm25_retriever= BM25Retriever.from_documents(all_splits)


Loaded 6361 total splits (5110 Django docs + 1251 Python codebase).


In [5]:
bm25_retriever.k=8

In [6]:
#Creating hybrid retriever
hybrid_retriever= EnsembleRetriever(
    retrievers=[django_retriever, python_retriever, bm25_retriever],
    weights=[0.4,0.3,0.3]#40% django sementic search, 30% python sementic and keyword
    
)

In [7]:
#Reranker block of code
reranker= CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1652.58it/s]


In [8]:
@tool
def retrieve_django_content(query: str) -> str:
    """
    Search the Django knowledge base for documentation, code examples,
    debugging information, and practical examples.
    Use this tool to gain more context and information needed to answer the users question
    """
    print("Using retriever......")
    docs = hybrid_retriever.invoke(query)

    print("Reranking......")
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )
    top_docs = scored_docs[:3]

    return "\n\n".join(doc.page_content for doc, score in top_docs)


def _get_bin_dir(env_path: Path) -> Path:
    """Returns the folder inside a venv where executables live (differs by OS)."""
    return env_path / "Scripts" if sys.platform == "win32" else env_path / "bin"


def _exe(bin_dir: Path, name: str) -> Path:
    """Adds .exe to the executable name only on Windows."""
    return bin_dir / (f"{name}.exe" if sys.platform == "win32" else name)


@tool
def create_virtualenv(directory: str) -> str:
    """Runs a command to create a new virtual environment with the given name in the specified directory."""
    print("Creating virtual environent......")
    target_path = Path(directory)
    target_path.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        [sys.executable, "-m", "venv", "env"],
        cwd=target_path, capture_output=True, text=True
    )
    if result.returncode != 0:
        return f"Failed to create virtual environment: {result.stderr}"
    return f"Created virtual environment env in {directory}"


def _install_django(directory: str) -> str:
    """Runs a command to install Django in the virtual environment."""
    print("Installing Django......")
    target_path = Path(directory)
    pip_exe = _exe(_get_bin_dir(target_path / "env"), "pip")
    result = subprocess.run(
        [str(pip_exe), "install", "django"],
        cwd=target_path, capture_output=True, text=True
    )
    if result.returncode != 0:
        return f"Failed to install Django: {result.stderr}"
    return "Installed Django in virtual environment env"


@tool
def create_django_project(name: str, directory: str) -> str:
    """Runs a command to create a new Django project with the given name in the specified directory."""
    print("Creating Django project......")
    target_path = Path(directory)
    target_path.mkdir(parents=True, exist_ok=True)
    bin_dir = _get_bin_dir(target_path / "env")
    pip_exe = _exe(bin_dir, "pip")
    django_admin = _exe(bin_dir, "django-admin")

    check = subprocess.run([str(pip_exe), "list"], capture_output=True, text=True)
    if "django" not in check.stdout.lower():
        install_result = _install_django(directory)
        if "Failed" in install_result:
            return install_result

    result = subprocess.run(
        [str(django_admin), "startproject", name, "."],
        cwd=target_path, capture_output=True, text=True
    )
    if result.returncode != 0:
        return f"Failed to create Django project: {result.stderr}"
    return f"Created Django project {name} in {directory}"


_active_server_process = None

@tool
def run_django_server(project_name: str, directory: str) -> str:
    """Runs a command to start the Django development server asynchronously."""
    print("Running Django server......")
    global _active_server_process

    target_path = Path(directory) / project_name
    if not (target_path / "manage.py").exists():
        target_path = Path(directory)

    manage_py = target_path / "manage.py"
    if not manage_py.exists():
        return f"Could not find manage.py in {target_path} — make sure the project was created first."

    python_exe = _exe(_get_bin_dir(target_path.parent / "env"), "python")
    if not python_exe.exists():
        python_exe = _exe(_get_bin_dir(target_path / "env"), "python")

    if _active_server_process and _active_server_process.poll() is None:
        _active_server_process.terminate()

    _active_server_process = subprocess.Popen(
    [str(python_exe), "manage.py", "runserver", "--noreload"],
    cwd=target_path
    )
    return f"Django server initiated in {target_path}"


@tool
def stop_django_server() -> str:
    """Stops the currently running Django development server task."""
    print("Stopping Django server......")
    global _active_server_process

    if _active_server_process and _active_server_process.poll() is None:
        _active_server_process.terminate()
        try:
            _active_server_process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            _active_server_process.kill()
        _active_server_process = None
        return "The Django server has been successfully stopped."

    return "No active Django server is currently running."

In [9]:
#Constructing Agent and memory
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
memory=MemorySaver()



In [10]:
#Trimming users messages to preserve context window
@wrap_model_call
def limit_history(request, handler):
    messages = request.state["messages"]

    recent_messages = messages[-2:]

    request.state["messages"] = recent_messages

    return handler(request)

In [11]:
agent= create_agent(
    model=llm,
    tools=[retrieve_django_content, create_virtualenv, create_django_project, run_django_server, stop_django_server, ],
    system_prompt="""
You are a Django engineering assistant with two jobs: reviewing Django code, and setting up/running Django projects using the tools available to you.

You have two categories of tools:

1. KNOWLEDGE TOOL — retrieve_django_content
   MANDATORY: You MUST call retrieve_django_content at least once for EVERY user request before doing anything else — before answering a question, before reviewing code, and before calling any action tool. This is not optional and does not depend on how confident you feel about the answer. Even for requests to create/run a project, call retrieve_django_content first (e.g. query it for "django project setup best practices" or something relevant to the specific request) to ground your approach, then proceed to the action tools.

2. ACTION TOOLS — these perform real actions on the user's machine:
   - create_virtualenv(directory): creates a virtual environment named "env" inside the given directory
   - create_django_project(name, directory): creates a new Django project (installs Django first if missing)
   - run_django_server(project_name, directory): starts the Django dev server in the background
   - stop_django_server(): stops the currently running dev server

STRICT ORDER OF OPERATIONS:
1. ALWAYS call retrieve_django_content first, no exceptions.
2. Then, based on what the user asked:
   - If it's a question/review → answer using the retrieved context.
   - If it's a setup/run request → proceed to the action tools in order: create_virtualenv → create_django_project → run_django_server (skip a step only if the user says it already exists).

RULES FOR ACTION TOOLS:
- If the user asks you to create, set up, scaffold, start, or run a Django project — use the tools. Do not just describe the steps in text; call the tools.
- Infer sensible defaults if the user is vague (e.g. project name "myproject", directory as an absolute path if none given — ask for one if you truly cannot infer it) and state the assumption in one short sentence before acting.
- After calling a tool, report what actually happened in one or two sentences — summarize, don't paste the tool's raw return string verbatim.
- If a tool call fails or the server doesn't start, say so plainly and suggest the likely cause — do not pretend it succeeded.
- Never call run_django_server without first confirming a project actually exists at that path.
- Only call stop_django_server if the user asks to stop, restart, or if you are about to start a new server and one may already be running.

RULES FOR CODE REVIEW / KNOWLEDGE ANSWERS:
- Answer using the context retrieved from retrieve_django_content. If it doesn't return anything relevant, say "I don't have this information in my knowledge base" rather than guessing.
- Keep review answers direct: lead with the answer, then 1-3 sentences of essential context.
- Use bullet points only when the user asks for a list of issues/items.
- Don't narrate your retrieval or tool-selection process to the user (no "I searched the knowledge base..." or "I'm now calling retrieve_django_content...") — just do it and report the outcome.

GENERAL:
- Be concise. No filler like "Based on the above" or "In conclusion."
- If a request is ambiguous between "explain how to do X" and "do X for me," default to doing it if action tools are available and relevant — ask only if the ambiguity would cause you to act on the wrong project/directory.
""",
    middleware=[limit_history],
    checkpointer=memory
)

#Memory
config = {"configurable": {"thread_id": "test-7"}}

In [15]:
#Creatin User Interface
while True:
    question= input("Any Question about django: ").strip()
    if question.lower() in ["exit", "quit", "q"]:
        print("See you soon...")
        break
    if not question:
        continue

    result= agent.invoke({
        "messages": [HumanMessage(content=question)]
    }, config=config)

    print("\n Final Result")
    print(result["messages"][-1].content)

Stopping Django server......

 Final Result
The Django development server has been stopped (no active server was running).

 Final Result
**Django project created**

- **inventory** – scaffolded in `C:\Users\ZENOID\Desktop\Home\home\self_made.projects\Standard Projects\code_debugger\inventory`.

**Issue(s) in the `get_products` view you provided**

- **Unnecessary console printing** – `print(p.category.name)` writes to the server console on every request; use proper logging or remove it.
- **Potential N+1 database queries** – Accessing `p.category.name` inside the loop triggers a separate query per product. Use `select_related('category')` on the queryset to fetch categories in one query.
- **Missing imports** – The view needs `from django.shortcuts import render` and an import for the `Product` model (`from .models import Product`); otherwise it will raise `NameError`.
- **No handling for empty querysets or missing template** – While not a runtime error, handling the empty case or ens